In [1]:
# 1. INSTALACIÓN E IMPORTACIÓN DE LIBRERÍAS
!pip install -q pandas numpy scikit-learn plotly gdown

import sqlite3
import pandas as pd
import numpy as np
import plotly.express as px
import gdown
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import joblib
from google.colab import files


# 2. CARGA DE LA BASE DE DATOS DESDE GOOGLE DRIVE
file_id = '1r86ErEdX2zoFfrhwM6IaVOV4d9aaHc9M'
url = f'https://drive.google.com/uc?id={file_id}'
db_filename = 'agrotech.db'

print("📥 Descargando la base de datos desde Google Drive...")
gdown.download(url, db_filename, quiet=False)


📥 Descargando la base de datos desde Google Drive...


Downloading...
From: https://drive.google.com/uc?id=1r86ErEdX2zoFfrhwM6IaVOV4d9aaHc9M
To: /content/agrotech.db
100%|██████████| 258k/258k [00:00<00:00, 48.3MB/s]


'agrotech.db'

In [2]:
# 3. EXTRACCIÓN Y PREPARACIÓN DE DATOS
conn = sqlite3.connect(db_filename)
query = """
    SELECT timestamp, nodo_id, temperatura, humedad,
           humedad_suelo, temperatura_suelo, ph, conductividad,
           nitrogeno, fosforo, potasio
    FROM mediciones
    WHERE temperatura IS NOT NULL AND humedad_suelo IS NOT NULL
"""
df = pd.read_sql_query(query, conn)
conn.close()

if len(df) < 50:
    print(f"⚠️ Advertencia: Tienes {len(df)} registros. El modelo requiere mínimo 50 para resultados fiables.")

# Definición de conjuntos de características
features_complete = [
    'temperatura', 'humedad', 'humedad_suelo', 'temperatura_suelo',
    'ph', 'conductividad', 'nitrogeno', 'fosforo', 'potasio'
]
features_no_npk = [
    'temperatura', 'humedad', 'humedad_suelo', 'temperatura_suelo',
    'ph', 'conductividad'
]

# Limpieza de datos (eliminando filas con valores nulos en el set completo)
X_complete = df[features_complete].dropna()
df_clean = df.loc[X_complete.index].copy()
X_no_npk = df_clean[features_no_npk]

# 4. CONFIGURACIÓN Y ENTRENAMIENTO DE LOS DOS MODELOS
# Ambos con los mismos parámetros de configuración
modelo_completo = IsolationForest(
    n_estimators=100,
    contamination=0.035,
    random_state=42
)
modelo_completo.fit(X_complete)

modelo_no_npk = IsolationForest(
    n_estimators=100,
    contamination=0.035,
    random_state=42
)
modelo_no_npk.fit(X_no_npk)

# 5. PREDICCIÓN Y CLASIFICACIÓN DE SEVERIDAD
# Modelo 1: Completo (Con NPK)
df_clean['Score_Completo'] = modelo_completo.decision_function(X_complete)
df_clean['Pred_Completo'] = modelo_completo.predict(X_complete)

# Modelo 2: Sin NPK
df_clean['Score_No_NPK'] = modelo_no_npk.decision_function(X_no_npk)
df_clean['Pred_No_NPK'] = modelo_no_npk.predict(X_no_npk)

def clasificar_estado(pred, score):
    if pred == 1:
        return 'Sana'
    elif score < -0.05:
        return 'Crítica'
    else:
        return 'Advertencia'

df_clean['Estado_Completo'] = df_clean.apply(lambda r: clasificar_estado(r['Pred_Completo'], r['Score_Completo']), axis=1)
df_clean['Estado_No_NPK'] = df_clean.apply(lambda r: clasificar_estado(r['Pred_No_NPK'], r['Score_No_NPK']), axis=1)

df_clean['Score_Completo'] = df_clean['Score_Completo'].round(4)
df_clean['Score_No_NPK'] = df_clean['Score_No_NPK'].round(4)



In [3]:

# Colores de severidad
color_map = {'Sana': '#2E7D32', 'Advertencia': '#F57F17', 'Crítica': '#C62828'}

# 6. GRÁFICAS COMPARATIVAS 2D (Temperatura vs Humedad del Suelo)
fig_2d_comp = px.scatter(
    df_clean,
    x='temperatura',
    y='humedad_suelo',
    color='Estado_Completo',
    color_discrete_map=color_map,
    hover_data=['timestamp', 'nodo_id', 'Score_Completo', 'ph', 'conductividad', 'nitrogeno', 'fosforo', 'potasio'],
    title="Modelo Completo (Con NPK) - Temperatura vs Humedad de Suelo",
    labels={'temperatura': 'Temperatura (°C)', 'humedad_suelo': 'Humedad Suelo (%)'}
)
fig_2d_comp.update_traces(marker=dict(size=8, opacity=0.8, line=dict(width=1, color='DarkSlateGrey')))
fig_2d_comp.show()

fig_2d_no_npk = px.scatter(
    df_clean,
    x='temperatura',
    y='humedad_suelo',
    color='Estado_No_NPK',
    color_discrete_map=color_map,
    hover_data=['timestamp', 'nodo_id', 'Score_No_NPK', 'ph', 'conductividad'],
    title="Modelo sin NPK - Temperatura vs Humedad de Suelo",
    labels={'temperatura': 'Temperatura (°C)', 'humedad_suelo': 'Humedad Suelo (%)'}
)
fig_2d_no_npk.update_traces(marker=dict(size=8, opacity=0.8, line=dict(width=1, color='DarkSlateGrey')))
fig_2d_no_npk.show()

In [4]:

# 7. GRÁFICA COMPARATIVA 3D (PCA)
# PCA para el Modelo Completo (9 dimensiones a 3)
scaler_c = StandardScaler()
X_scaled_c = scaler_c.fit_transform(X_complete)
pca_c = PCA(n_components=3)
comp_c = pca_c.fit_transform(X_scaled_c)

df_clean['Eje_C1'] = comp_c[:, 0]
df_clean['Eje_C2'] = comp_c[:, 1]
df_clean['Eje_C3'] = comp_c[:, 2]

fig_3d_comp = px.scatter_3d(
    df_clean,
    x='Eje_C1', y='Eje_C2', z='Eje_C3',
    color='Estado_Completo',
    color_discrete_map=color_map,
    hover_data={
        'Eje_C1': False, 'Eje_C2': False, 'Eje_C3': False,
        'Score_Completo': True, 'temperatura': True, 'humedad': True,
        'humedad_suelo': True, 'ph': True, 'conductividad': True,
        'nitrogeno': True, 'fosforo': True, 'potasio': True
    },
    title="PCA Modelo Completo (9 Dimensiones reducidas a 3)"
)
fig_3d_comp.update_traces(marker=dict(size=5, opacity=0.8, line=dict(width=0)))
fig_3d_comp.update_layout(scene=dict(xaxis_title="CP 1", yaxis_title="CP 2", zaxis_title="CP 3"), margin=dict(l=0, r=0, b=0, t=40))
fig_3d_comp.show()

# PCA para el Modelo Sin NPK (6 dimensiones a 3)
scaler_n = StandardScaler()
X_scaled_n = scaler_n.fit_transform(X_no_npk)
pca_n = PCA(n_components=3)
comp_n = pca_n.fit_transform(X_scaled_n)

df_clean['Eje_N1'] = comp_n[:, 0]
df_clean['Eje_N2'] = comp_n[:, 1]
df_clean['Eje_N3'] = comp_n[:, 2]

fig_3d_no_npk = px.scatter_3d(
    df_clean,
    x='Eje_N1', y='Eje_N2', z='Eje_N3',
    color='Estado_No_NPK',
    color_discrete_map=color_map,
    hover_data={
        'Eje_N1': False, 'Eje_N2': False, 'Eje_N3': False,
        'Score_No_NPK': True, 'temperatura': True, 'humedad': True,
        'humedad_suelo': True, 'ph': True, 'conductividad': True
    },
    title="PCA Modelo sin NPK (6 Dimensiones reducidas a 3)"
)
fig_3d_no_npk.update_traces(marker=dict(size=5, opacity=0.8, line=dict(width=0)))
fig_3d_no_npk.update_layout(scene=dict(xaxis_title="CP 1", yaxis_title="CP 2", zaxis_title="CP 3"), margin=dict(l=0, r=0, b=0, t=40))
fig_3d_no_npk.show()


In [5]:
# 8. RESUMEN COMPARATIVO
print("📊 Resumen de Clasificación - Modelo Completo:")
print(df_clean['Estado_Completo'].value_counts())
print("\n📊 Resumen de Clasificación - Modelo sin NPK:")
print(df_clean['Estado_No_NPK'].value_counts())


📊 Resumen de Clasificación - Modelo Completo:
Estado_Completo
Sana           231
Crítica          8
Advertencia      1
Name: count, dtype: int64

📊 Resumen de Clasificación - Modelo sin NPK:
Estado_No_NPK
Sana           231
Crítica          5
Advertencia      4
Name: count, dtype: int64


In [6]:

# Guardar los modelos en archivos .joblib
joblib.dump(modelo_completo, 'modelo_completo.joblib')
joblib.dump(modelo_no_npk, 'modelo_no_npk.joblib')

print("✅ Modelos exportados correctamente.")

# Descargar los archivos automáticamente en el navegador
#files.download('modelo_completo.joblib')
#files.download('modelo_no_npk.joblib')

#Cambiarles el nombre a "iforest_model.joblib" para que el algoritmo lo reconozca.

✅ Modelos exportados correctamente.


In [7]:
# PCA para el Modelo Completo (9 dimensiones a 3)
scaler_c = StandardScaler()
X_scaled_c = scaler_c.fit_transform(X_complete)
pca_c = PCA(n_components=3)
comp_c = pca_c.fit_transform(X_scaled_c)

df_clean['Eje_C1'] = comp_c[:, 0]
df_clean['Eje_C2'] = comp_c[:, 1]
df_clean['Eje_C3'] = comp_c[:, 2]

fig_3d_comp = px.scatter_3d(
    df_clean,
    x='Eje_C1', y='Eje_C2', z='Eje_C3',
    color='Estado_Completo',
    color_discrete_map=color_map,
    hover_data={
        'Eje_C1': False, 'Eje_C2': False, 'Eje_C3': False,
        'Score_Completo': True, 'temperatura': True, 'humedad': True,
        'humedad_suelo': True, 'ph': True, 'conductividad': True,
        'nitrogeno': True, 'fosforo': True, 'potasio': True
    },
    title="PCA Modelo Completo (9 Dimensiones reducidas a 3)"
)
fig_3d_comp.update_traces(marker=dict(size=5, opacity=0.8, line=dict(width=0)))
fig_3d_comp.update_layout(scene=dict(xaxis_title="CP 1", yaxis_title="CP 2", zaxis_title="CP 3"), margin=dict(l=0, r=0, b=0, t=40))
fig_3d_comp.show()